***Получение прогноза для тестовых данных***

Распакуем наши обработчики

In [161]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mamoUtils as mu
np.__version__, pd.__version__

('2.5.5', '3.1.1')

In [162]:
import joblib

pipeline = joblib.load('models/best_model.pkl')

bestForest_1 = pipeline['model_1']
bestForest_2 = pipeline['model_2']
bestForest_3 = pipeline['model_3']
yearsScaler = pipeline['years_scaler']
monthEncoder = pipeline['month_encoder']
unitEncoder = pipeline['unit_encoder']
skuEncoder = pipeline['sku_encoder']
scaler_1 = pipeline['param_scaler_1']
scaler_2 = pipeline['param_scaler_2']
scaler_3 = pipeline['param_scaler_3']

Далее загружаем из файла данные для прогноза

In [163]:
testDataFrame = pd.read_csv('data/test.csv')
testDataFrame.head()

,id,sku,unit,month
0,0_2021-12,0,ШТ,2021-12
1,0_2022-01,0,ШТ,2022-01
2,0_2022-02,0,ШТ,2022-02
3,1_2021-12,1,ШТ,2021-12
4,1_2022-01,1,ШТ,2022-01


In [164]:
fillDataFrame = pd.read_csv('data/train.csv')
fillDataFrame.head()

,sku,unit,2019-01,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,...,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11
0,0,ШТ,128.0,125.0,179.0,199.0,204.0,246.0,172.0,235.0,...,379.0,250.0,317.0,172.0,193.0,173.0,121.0,318.0,328.0,208.0
1,1,ШТ,65.0,64.0,54.0,91.0,74.0,91.0,138.0,115.0,...,68.0,65.0,72.0,82.0,85.0,69.0,75.0,100.0,72.0,89.0
2,2,ШТ,157.0,205.0,210.0,205.0,265.0,248.0,292.0,268.0,...,334.0,380.0,409.0,309.0,388.0,438.0,404.0,293.0,274.0,381.0
3,3,ШТ,2450.0,2280.0,2144.0,2283.0,2177.0,2888.0,3066.0,2288.0,...,2806.0,2085.0,1984.0,2086.0,2871.0,2204.0,1821.0,2187.0,1672.0,1665.0
4,4,ШТ,119.0,154.0,154.0,270.0,513.0,718.0,720.0,292.0,...,222.0,258.0,289.0,662.0,759.0,616.0,365.0,321.0,392.0,409.0


In [165]:
splitter = mu.SplitDateOnColumn()
splittedData = splitter.transform(fillDataFrame)
dateSplitter = mu.SplitDateOnYearsMonth()
transformedFillData = dateSplitter.transform(splittedData)
transformedFillData.loc[transformedFillData['resultValue'] < 0, 'resultValue'] = 0
print(len(transformedFillData.loc[transformedFillData['resultValue'] < 0, 'resultValue']))
transformedFillData.head(40)

0


,sku,unit,resultValue,years,month
0,0,ШТ,128.0,2019,1
1,0,ШТ,125.0,2019,2
2,0,ШТ,179.0,2019,3
3,0,ШТ,199.0,2019,4
4,0,ШТ,204.0,2019,5
5,0,ШТ,246.0,2019,6
6,0,ШТ,172.0,2019,7
7,0,ШТ,235.0,2019,8
8,0,ШТ,238.0,2019,9
9,0,ШТ,228.0,2019,10


In [166]:
testDataFrameDates = dateSplitter.transform(testDataFrame)
testDataFrameDates.head()

,id,sku,unit,years,month
0,0_2021-12,0,ШТ,2021,12
1,0_2022-01,0,ШТ,2022,1
2,0_2022-02,0,ШТ,2022,2
3,1_2021-12,1,ШТ,2021,12
4,1_2022-01,1,ШТ,2022,1


In [167]:
ids = testDataFrameDates[['id']]
testDataFrameDates = testDataFrameDates.drop('id', axis = 1)
OneMonthDataFrame = testDataFrameDates.loc[(testDataFrameDates['years'] == 2021) & (testDataFrameDates['month'] == 12)]
TwoMonthDataFrame = testDataFrameDates.loc[(testDataFrameDates['years'] == 2022) & (testDataFrameDates['month'] == 1)]
ThreeMonthDataFrame = testDataFrameDates.loc[(testDataFrameDates['years'] == 2022) & (testDataFrameDates['month'] == 2)]

In [168]:
def get_lag_for_date(dateFrame, lagsFrom, lags):
    df = dateFrame.copy()
    columns = []
    for i in lags:
        lag_string = f'lag_{i}'
        columns += [lag_string]
        targetYear = dateFrame.iloc[0]['years']
        targetMonth = dateFrame.iloc[0]['month'] - i;
        while(targetMonth <= 0):
            targetYear -= 1
            targetMonth += 12
        lagVal = lagsFrom.loc[(lagsFrom['years'] == targetYear) & (lagsFrom['month'] == targetMonth), ['sku','resultValue']]
        lagVal = lagVal.rename(columns={'resultValue': lag_string})
        df = df.merge(lagVal, on='sku', how='left')
    return (df, columns)
def get_rolling_mean_for_date(dateFrame, rollFrom, startYear, startMonth, windows):
    df = dateFrame.copy()
    columns = []
    for i in windows:
        rm_str = f'rolling_mean_{i}'
        columns += [rm_str]
        resValForMean = pd.DataFrame(columns = rollFrom.columns)
        for k in range(0, i):
            concreteYear = startYear
            concreteMonth = startMonth - k
            while(concreteMonth <= 0):
                concreteYear -= 1
                concreteMonth += 12
            saveDf = rollFrom.loc[(rollFrom['years'] == concreteYear) & (rollFrom['month'] == concreteMonth)]
            resValForMean = pd.concat([resValForMean, saveDf])
        mean = resValForMean.groupby('sku')[['resultValue']].mean()
        mean = mean.rename(columns={'resultValue': rm_str})
        df = df.merge(mean, on='sku', how='left')
    return (df, columns)
def get_rolling_std_for_date(dateFrame, rollFrom, startYear, startMonth, windows):
    df = dateFrame.copy()
    columns = []
    for i in windows:
        rs_str = f'rolling_std_{i}'
        columns += [rs_str]
        resValForMean = pd.DataFrame(columns = rollFrom.columns)
        for k in range(0, i):
            concreteYear = startYear
            concreteMonth = startMonth - k
            while(concreteMonth <= 0):
                concreteYear -= 1
                concreteMonth += 12
            saveDf = rollFrom.loc[(rollFrom['years'] == concreteYear) & (rollFrom['month'] == concreteMonth)]
            resValForMean = pd.concat([resValForMean, saveDf])
        std = resValForMean.groupby('sku')[['resultValue']].std()
        std = std.rename(columns={'resultValue': rs_str})
        df = df.merge(std, on='sku', how='left')
    return (df, columns)
def get_season_ratio(dateFrame, ratioFrom, lag):
    df = dateFrame.copy()
    month = dateFrame['month'].iloc[0]
    season_str = 'season_ratio'
    seasonMean = ratioFrom.groupby(['sku', 'month'])['resultValue'].mean()
    filteredMean = seasonMean[seasonMean.index.get_level_values('month').isin([month])];
    filteredMean = filteredMean.rename(season_str)
    df = df.merge(filteredMean, on='sku', how='left')
    df[season_str] = dateFrame[lag] / (df[season_str] + 1);
    return (df, [season_str])
def get_season_deviation(dateFrame, ratioFrom, lag):
    df = dateFrame.copy()
    month = dateFrame['month'].iloc[0]
    season_str = 'season_deviation'
    seasonMean = ratioFrom.groupby(['sku', 'month'])['resultValue'].mean()
    filteredMean = seasonMean[seasonMean.index.get_level_values('month').isin([month])];
    filteredMean = filteredMean.rename(season_str)
    df = df.merge(filteredMean, on='sku', how='left')
    df[season_str] = dateFrame[lag] - df[season_str];
    return (df, [season_str])
def get_dataFrame_with_params(dataFrame, fillDataFrame, lags, yearsFromWindows, monthFromWindows, windows):
    baseIndexes = dataFrame.index
    df = dataFrame.copy()
    df, lag_columns = get_lag_for_date(df, fillDataFrame, lags)
    df, mean_columns = get_rolling_mean_for_date(df, fillDataFrame, yearsFromWindows, monthFromWindows, windows)
    df, std_columns = get_rolling_std_for_date(df, fillDataFrame, yearsFromWindows, monthFromWindows, windows)
    df, season_columns = get_season_ratio(df, fillDataFrame, lag_columns[0])
    df, season_dev_columns = get_season_deviation(df, fillDataFrame, lag_columns[0])
    df.index = baseIndexes
    return (df, lag_columns + mean_columns + std_columns + season_dev_columns, season_columns)

data_1, columns_1, freeColumn_1 = get_dataFrame_with_params(OneMonthDataFrame, transformedFillData, [1, 2, 3, 12], 2021, 11, [3, 6, 12])
data_1.head()

,sku,unit,years,month,lag_1,lag_2,lag_3,lag_12,rolling_mean_3,rolling_mean_6,rolling_mean_12,rolling_std_3,rolling_std_6,rolling_std_12,season_ratio,season_deviation
0,0,ШТ,2021,12,208.0,328.0,318.0,271.0,284.666667,223.5,255.416667,66.583281,82.556042,81.362777,0.960739,-7.5
3,1,ШТ,2021,12,89.0,72.0,100.0,80.0,87.0,81.666667,75.333333,14.106736,11.826524,13.418666,1.112500,10.0
6,2,ШТ,2021,12,381.0,274.0,293.0,327.0,316.0,363.0,355.833333,57.087652,64.924572,51.183863,1.465385,122.0
9,3,ШТ,2021,12,1665.0,1672.0,2187.0,1625.0,1841.333333,2070.0,2128.583333,299.376574,459.929560,424.497448,0.828564,-343.5
12,4,ШТ,2021,12,409.0,392.0,321.0,599.0,374.0,477.0,426.083333,46.679760,171.798719,185.726807,0.783525,-112.0


In [169]:
def scale_data(dataFrame, scaledColumns, freeColumns, yearsScaler, paramScaler, monthEncoder, unitEncoder, skuEncoder):
    encoded_data = encode_data(dataFrame, unitEncoder, skuEncoder)
    scaledDate = pd.DataFrame(yearsScaler.transform(dataFrame[['years']]), index = dataFrame.index, columns = ['years'])
    encodedMonth = monthEncoder.transform(dataFrame[['month']]).toarray()
    encodedMonthDF = pd.DataFrame(encodedMonth, index = dataFrame.index, columns = monthEncoder.get_feature_names_out(input_features=dataFrame[['month']].columns))
    scaledParam = pd.DataFrame(
        paramScaler.transform(
            encoded_data[['sku']].join(
                dataFrame[scaledColumns]
            )
        ), index = dataFrame.index,
        columns = ['sku'] + scaledColumns)
    resultDataFrame = scaledParam.join(encoded_data.drop('sku', axis=1)).join(encodedMonthDF).join(scaledDate).join(dataFrame[freeColumns])
    return resultDataFrame

def encode_data(dataFrame, unitEncoder, skuEncoder):
    encodedSku = pd.DataFrame(skuEncoder.transform(dataFrame[['sku']].astype(str)), index = dataFrame.index, columns = ['sku'])
    encodedUnit = unitEncoder.transform(dataFrame[['unit']]).toarray()
    encodedUnitDF = pd.DataFrame(encodedUnit, index = dataFrame.index, columns = unitEncoder.get_feature_names_out(input_features=dataFrame[['unit']].columns))
    return encodedSku.join(encodedUnitDF)

scaled_X_test_1 = scale_data(data_1, columns_1, freeColumn_1, yearsScaler, scaler_1, monthEncoder, unitEncoder, skuEncoder)

In [170]:
scaled_X_test_1.head()

,sku,lag_1,lag_2,lag_3,lag_12,rolling_mean_3,rolling_mean_6,rolling_mean_12,rolling_std_3,rolling_std_6,...,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12,years,season_ratio
0,-0.171858,-0.106557,0.063978,0.051604,0.004329,-0.010879,-0.114236,-0.094220,0.026568,-0.044969,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.960739
3,-0.313331,-0.269126,-0.292072,-0.252441,-0.271284,-0.279692,-0.300828,-0.321210,-0.262506,-0.308616,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.112500
6,-0.060240,0.129781,-0.011127,0.016736,0.085137,0.031732,0.069287,0.032352,-0.025740,-0.110691,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.465385
9,2.370285,1.883880,1.933241,2.658298,1.958153,2.106074,2.314970,2.266852,1.308940,1.361704,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.828564
12,0.093700,0.168033,0.152990,0.055788,0.477633,0.110607,0.219262,0.120900,-0.083073,0.287686,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.783525


In [171]:
predict_1 = pd.Series(bestForest_1.predict(scaled_X_test_1), index = scaled_X_test_1.index)

In [172]:
predict_1

0        219.401996
3         86.595809
6        308.954722
9       1896.062694
12       456.414156
           ...     
4953    1563.634256
4956     372.129827
4959    1833.880960
4962     985.261131
4965     131.531291
Length: 1656, dtype: float64

Получили первый прогноз, самое время получать и второй и третий

In [173]:
data_2, columns_2, freeColumn_2 = get_dataFrame_with_params(TwoMonthDataFrame, transformedFillData, [2, 3, 4, 12], 2021, 11, [3, 6, 12])
scaled_X_test_2 = scale_data(data_2, columns_2, freeColumn_2, yearsScaler, scaler_2, monthEncoder, unitEncoder, skuEncoder)
predict_2 = pd.Series(bestForest_2.predict(scaled_X_test_2), index = scaled_X_test_2.index)

In [174]:
data_3, columns_3, freeColumn_3 = get_dataFrame_with_params(ThreeMonthDataFrame, transformedFillData, [3, 12, 13, 24], 2021, 11, [3, 6, 12])
scaled_X_test_3 = scale_data(data_3, columns_3, freeColumn_3, yearsScaler, scaler_3, monthEncoder, unitEncoder, skuEncoder)
predict_3 = pd.Series(bestForest_3.predict(scaled_X_test_3), index = scaled_X_test_3.index)

In [175]:
allPrediction = pd.concat([predict_1, predict_2, predict_3]).sort_index()
allPrediction

0       219.401996
1       209.864147
2       222.540394
3        86.595809
4        70.395422
           ...    
4963    779.542594
4964    891.853096
4965    131.531291
4966    121.367035
4967    110.186124
Length: 4968, dtype: float64

In [176]:
result = ids.copy()
result['prediction'] = allPrediction
result

,id,prediction
0,0_2021-12,219.401996
1,0_2022-01,209.864147
2,0_2022-02,222.540394
3,1_2021-12,86.595809
4,1_2022-01,70.395422
...,...,...
4963,1654_2022-01,779.542594
4964,1654_2022-02,891.853096
4965,1655_2021-12,131.531291
4966,1655_2022-01,121.367035


In [177]:
result.to_csv('data/predict.csv', index=False)